#### Load the cleaned data from data/processed directory

In [9]:
import pandas as pd

df = pd.read_csv('../data/processed/flights_cleaned.csv')

print("Shape:", df.shape)
df.head()

Shape: (300259, 12)


,airline,from,to,price,stops_numeric,duration_minutes,departure_hour,arrival_hour,month,day,is_weekend,is_business
0,Air India,Delhi,Mumbai,25612,0,120,18,20,2,4,0,1
1,Air India,Delhi,Mumbai,25612,0,135,19,21,2,4,0,1
2,Air India,Delhi,Mumbai,42220,1,1485,20,20,2,4,0,1
3,Air India,Delhi,Mumbai,44450,1,1590,21,23,2,4,0,1
4,Air India,Delhi,Mumbai,46690,1,400,17,23,2,4,0,1


#### Split data onto train(64%), validation(16%), test(20%) 
##### for sure before splitting we must drop the target column(price) 
##### X contains all cleaned_flight data without price
##### Y just contains price
##### then split these to X_train, Y_train, X_val, Y_val, X_test, Y_test   

In [4]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['price'])
y = df['price']

# First split off test set
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Then split remaining into train and validation
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.2,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_val shape:  ", X_val.shape)
print("X_test shape: ", X_test.shape)

X_train shape: (192165, 11)
X_val shape:   (48042, 11)
X_test shape:  (60052, 11)


#### Here we used something called Target Encoding 
##### this type of encoding is used to not add many new columns
##### so here we are encoding based on the average price of each category 
##### Note: when we encode the test and val set we encode them from means of training data to avoid data leaking   

In [5]:
cat_cols = ['airline', 'from', 'to']

# Calculate means from TRAINING data only
for col in cat_cols:
    means = X_train[col].map(y_train.groupby(X_train[col]).mean())
    X_train[col + '_encoded'] = means
    X_val[col + '_encoded']   = X_val[col].map(y_train.groupby(X_train[col]).mean())
    X_test[col + '_encoded']  = X_test[col].map(y_train.groupby(X_train[col]).mean())

##### Drop original string columns

In [6]:
X_train = X_train.drop(columns=cat_cols)
X_val   = X_val.drop(columns=cat_cols)
X_test  = X_test.drop(columns=cat_cols)

In [7]:
print("X_train columns:", X_train.columns.tolist())
print("\nAny nulls in X_train:", X_train.isnull().sum().sum())
print("Any nulls in X_val:  ", X_val.isnull().sum().sum())
print("Any nulls in X_test: ", X_test.isnull().sum().sum())

X_train columns: ['stops_numeric', 'duration_minutes', 'departure_hour', 'arrival_hour', 'month', 'day', 'is_weekend', 'is_business', 'airline_encoded', 'from_encoded', 'to_encoded']

Any nulls in X_train: 0
Any nulls in X_val:   0
Any nulls in X_test:  0


#### Save everything in memory

In [8]:
import os

os.makedirs('../data/processed', exist_ok=True)

X_train.to_csv('../data/processed/X_train.csv', index=False)
X_val.to_csv('../data/processed/X_val.csv',     index=False)
X_test.to_csv('../data/processed/X_test.csv',   index=False)

y_train.to_csv('../data/processed/y_train.csv', index=False)
y_val.to_csv('../data/processed/y_val.csv',     index=False)
y_test.to_csv('../data/processed/y_test.csv',   index=False)

print("All splits saved successfully!")

All splits saved successfully!
